<a href="https://colab.research.google.com/github/madelsu/Dynamic_Hyperparameter_Optimization_LLM/blob/main/(a)SQL_Database/SQL_Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

excel_path = r"...xlsx"
txt_path   = r"...txt"

# Read the Excel sheet
df = pd.read_excel(excel_path, sheet_name="ICSR_assessment", header=0)

# Save as tab-delimited text
df.to_csv(txt_path, sep="\t", index=False)

print(f"✅ Saved tab-delimited TXT at: {txt_path}")

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.types import Text, String

# -----------------------------
# 1) Path to your TXT file
# -----------------------------
txt_path = r"...txt"

# -----------------------------
# 2) Load the tab-delimited TXT
# -----------------------------
df = pd.read_csv(txt_path, sep="\t")

# -----------------------------
# 3) Clean column names for MySQL compatibility
# -----------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True)
      .str.replace(r'__+', '_', regex=True)
      .str.strip('_')
)

# -----------------------------
# 4) Connect to MySQL (XAMPP on port ...)
# -----------------------------
engine = create_engine("mysql+mysqlconnector://root:@.../db_icsr_assessment_manuela")

# -----------------------------
# 5) Define column types (long text columns → TEXT)
# -----------------------------
dtype_map = {}
for col in df.columns:
    if 'narrative' in col or 'reasoning' in col or 'text' in col:
        dtype_map[col] = Text()
    elif col.endswith('_id'):
        dtype_map[col] = String(64)

# -----------------------------
# 6) Write DataFrame to SQL in chunks
# -----------------------------
table_name = "icsr_assessment_import"

df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",   # ⚠️ overwrite table if exists; use 'append' later
    index=False,
    dtype=dtype_map,
    chunksize=500          # ✅ insert in batches of 500 rows
)

# -----------------------------
# 7) Verify row count
# -----------------------------
with engine.connect() as conn:
    rowcount = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
    print(f"✅ Loaded {rowcount} rows into {table_name}.")